<a href="https://colab.research.google.com/github/francesc0patti/qml-project/blob/main/Settimana1_Intro_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Importiamo le librerie fondamentali per manipolare dati e grafici
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Importiamo la libreria scikit-learn che contiene i dataset e i modelli classici
from sklearn import datasets

# 1. Carichiamo il dataset Iris
iris = datasets.load_iris()

# 2. Mettiamo i dati in una "tabella" Pandas per renderli leggibili (simile a Excel)
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)

# Aggiungiamo la colonna con le "etichette" (le specie di fiori che dovremo imparare a prevedere)
df['target'] = iris.target

# 3. Mostriamo le prime 5 righe della nostra tabella per verificare che tutto funzioni
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 54.6 MB/s eta 0:00:00
  Attempting uninstall: autograd
    Found existing installation: autograd 1.9.1
    Uninstalling autograd-1.9.1:
      Successfully uninstalled autograd-1.9.1


In [ ]:
import pennylane as qml
from pennylane import numpy as np
from sklearn import datasets
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1. PREPARAZIONE DATI
# ==========================================
iris = datasets.load_iris()
X = iris.data
Y = iris.target

# Il One-Hot Encoding ora ha 3 colonne (perché le nostre classi reali sono 3)
Y_one_hot = np.zeros((len(Y), 3))
for i in range(len(Y)):
    Y_one_hot[i, Y[i]] = 1.0

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# ==========================================
# 2. IL CIRCUITO A 16 STATI
# ==========================================
dev = qml.device("default.qubit", wires=4)

@qml.qnode(dev)
def circuito_completo(dati_input, pesi):
    qml.AngleEmbedding(features=dati_input, wires=range(4), rotation='Y')
    qml.BasicEntanglerLayers(weights=pesi, wires=range(4))

    # LA TUA IDEA: Misuriamo TUTTI e 4 i Qubit.
    # Questo restituisce un array di 16 probabilità (da |0000> a |1111>)
    return qml.probs(wires=range(4))


# ==========================================
# 3. MAPPATURA RIDONDANTE (Error Mitigation)
# ==========================================
def aggrega_probabilita(probs_16):
    # Suddividiamo i 16 stati del circuito nei 3 "bacini" delle nostre specie
    # Usiamo np.sum per sommare le onde di probabilità

    prob_setosa = np.sum(probs_16[0:5])       # Stati da 0 a 4 (5 stati)
    prob_versicolor = np.sum(probs_16[5:10])  # Stati da 5 a 9 (5 stati)
    prob_virginica = np.sum(probs_16[10:16])  # Stati da 10 a 15 (6 stati)

    # Restituiamo le 3 probabilità consolidate tramite np.stack
    # (indispensabile per far calcolare le derivate a PennyLane)
    return np.stack([prob_setosa, prob_versicolor, prob_virginica])

def calcola_errore(pesi, X, Y_oh):
    previsioni_aggregate = []

    for x in X:
        # 1. Il circuito calcola le probabilità sui 16 stati
        probs_16 = circuito_completo(x, pesi)

        # 2. Intercettiamo le probabilità e le raggruppiamo
        probs_3 = aggrega_probabilita(probs_16)
        previsioni_aggregate.append(probs_3)

    previsioni_aggregate = np.stack(previsioni_aggregate)
    return np.mean((Y_oh - previsioni_aggregate) ** 2)


# ==========================================
# 4. CICLO DI ADDESTRAMENTO
# ==========================================
ottimizzatore = qml.AdamOptimizer(stepsize=0.05)
pesi_quantistici = np.random.uniform(low=-np.pi, high=np.pi, size=(6, 4))
pesi_quantistici = np.tensor(pesi_quantistici, requires_grad=True)

print("Inizio addestramento con Mappatura Ridondante (16 stati -> 3 Classi)...\n")

for epoca in range(150):
    pesi_quantistici, errore = ottimizzatore.step_and_cost(lambda p: calcola_errore(p, X_scaled, Y_one_hot), pesi_quantistici)
    print(f"Epoca {epoca+1:2d} | Errore: {errore:.4f}")

print("\nAddestramento completato! Il modello ora usa bacini di stati tolleranti al rumore.")

Inizio addestramento con Mappatura Ridondante (16 stati -> 3 Classi)...

Epoca  1 | Errore: 0.2222
Epoca  2 | Errore: 0.2058
Epoca  3 | Errore: 0.1920
Epoca  4 | Errore: 0.1825
Epoca  5 | Errore: 0.1766
Epoca  6 | Errore: 0.1729
Epoca  7 | Errore: 0.1704
Epoca  8 | Errore: 0.1683
Epoca  9 | Errore: 0.1662
Epoca 10 | Errore: 0.1643
Epoca 11 | Errore: 0.1625
Epoca 12 | Errore: 0.1608
Epoca 13 | Errore: 0.1590
Epoca 14 | Errore: 0.1572
Epoca 15 | Errore: 0.1556
Epoca 16 | Errore: 0.1541
Epoca 17 | Errore: 0.1528
Epoca 18 | Errore: 0.1517
Epoca 19 | Errore: 0.1506
Epoca 20 | Errore: 0.1496
Epoca 21 | Errore: 0.1486
Epoca 22 | Errore: 0.1478
Epoca 23 | Errore: 0.1473
Epoca 24 | Errore: 0.1470
Epoca 25 | Errore: 0.1471
Epoca 26 | Errore: 0.1472
Epoca 27 | Errore: 0.1472
Epoca 28 | Errore: 0.1472
Epoca 29 | Errore: 0.1472
Epoca 30 | Errore: 0.1471
Epoca 31 | Errore: 0.1470
Epoca 32 | Errore: 0.1467
Epoca 33 | Errore: 0.1464
Epoca 34 | Errore: 0.1461
Epoca 35 | Errore: 0.1457
Epoca 36 | Errore

In [ ]:
np.save("pesi_iris_quantistici.npy", pesi_quantistici)

In [ ]:
pesi_caricati = np.load("pesi_iris_quantistici.npy")

# PennyLane richiede che diciamo esplicitamente se questi pesi possono essere modificati
pesi_quantistici = np.tensor(pesi_caricati, requires_grad=True)

In [ ]:
# ==========================================
# 5. TEST E ACCURATEZZA
# ==========================================

def calcola_accuratezza(pesi, X, Y_reali):
    previsioni_corrette = 0

    for i in range(len(X)):
        # 1. Il circuito sputa le 16 probabilità
        probs_16 = circuito_completo(X[i], pesi)

        # 2. Le raggruppiamo nei 3 bacini (le 3 specie)
        probs_3 = aggrega_probabilita(probs_16)

        # 3. La IA decide: la classe con la probabilità più alta vince
        classe_predetta = np.argmax(probs_3)

        # 4. Controlliamo se ha indovinato
        if classe_predetta == Y_reali[i]:
            previsioni_corrette += 1

    accuratezza = (previsioni_corrette / len(X)) * 100
    return accuratezza

# Calcoliamo l'accuratezza finale
acc = calcola_accuratezza(pesi_quantistici, X_scaled, Y)
print(f"L'accuratezza finale del modello quantistico è: {acc:.2f}%")